In [0]:
# Challenge 6 Setup: Production-Ready Dashboard
# Creates Gold tables and a published dashboard for the challenge tasks.

import re
import json
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType, TimestampType
from datetime import date, datetime, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"challenge_6_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- gold_sales: 1500 rows ---
random.seed(66)
spark.sql("DROP TABLE IF EXISTS gold_sales")

regions = ['Northeast', 'Southeast', 'Midwest', 'West', 'Northwest']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Food & Beverage']
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
channels = ['Online', 'Retail Store', 'Partner', 'Mobile App']
products = [
    'Laptop Pro', 'Tablet Air', 'Smart Watch', 'Wireless Earbuds',
    'Running Shoes', 'Denim Jacket', 'Standing Desk', 'LED Lamp',
    'Yoga Mat', 'Protein Bars', 'Coffee Beans', 'Tennis Racket'
]

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("net_revenue", DoubleType(), False),
    StructField("cost", DoubleType(), False)
])

rows = []
for i in range(1, 1501):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    unit_price = round(random.uniform(15, 500), 2)
    qty = random.randint(1, 5)
    discount = random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15])
    net_rev = round(qty * unit_price * (1 - discount), 2)
    cost_val = round(net_rev * random.uniform(0.4, 0.7), 2)
    rows.append((
        i, order_date, random.choice(regions), random.choice(products),
        random.choice(categories), random.choice(segments),
        random.choice(channels), qty, net_rev, cost_val
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("gold_sales")
print(f"Created gold_sales: {df.count()} rows")

In [0]:
# --- gold_customers: 100 rows ---
spark.sql("DROP TABLE IF EXISTS gold_customers")

random.seed(77)
first_names = ['James', 'Mary', 'Robert', 'Patricia', 'John', 'Jennifer', 'Michael', 'Linda', 'David', 'Sarah']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Wilson', 'Taylor']

cust_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("region", StringType(), False),
    StructField("signup_date", DateType(), False),
    StructField("lifetime_orders", IntegerType(), False),
    StructField("lifetime_revenue", DoubleType(), False)
])

cust_rows = []
for i in range(1, 101):
    cust_rows.append((
        i,
        f"{random.choice(first_names)} {random.choice(last_names)}",
        random.choice(segments),
        random.choice(regions),
        date(2020, 1, 1) + timedelta(days=random.randint(0, 1460)),
        random.randint(1, 40),
        round(random.uniform(100, 15000), 2)
    ))

df_cust = spark.createDataFrame(cust_rows, schema=cust_schema)
df_cust.write.saveAsTable("gold_customers")
print(f"Created gold_customers: {df_cust.count()} rows")

In [0]:
# --- Create and publish a dashboard ---
# Students will reference this dashboard_id in their audit queries.

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
parent_path = f"/Users/{username}"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
table_ref = f"`{catalog}`.`{schema}`.`gold_sales`"

dashboard_def = {
    "pages": [{
        "name": "page_main", "displayName": "Sales Overview",
        "layout": [
            {"widget": {"name": "w_title", "textbox_spec": "# Challenge 6: Production Dashboard\n\nPractice publishing, scheduling, and monitoring."}, "position": {"x": 0, "y": 0, "width": 6, "height": 1}},
            {"widget": {"name": "w_revenue", "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi", "fields": [{"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}], "spec": {"version": 3, "widgetType": "counter", "encodings": {"value": {"fieldName": "total_revenue", "displayName": "Total Revenue"}}}}, "position": {"x": 0, "y": 1, "width": 3, "height": 2}},
            {"widget": {"name": "w_orders", "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi", "fields": [{"name": "total_orders", "expression": "`total_orders`"}], "disaggregated": True}}], "spec": {"version": 3, "widgetType": "counter", "encodings": {"value": {"fieldName": "total_orders", "displayName": "Total Orders"}}}}, "position": {"x": 3, "y": 1, "width": 3, "height": 2}}
        ]
    }],
    "datasets": [{"name": "ds_kpi", "displayName": "KPI Summary", "query": f"SELECT ROUND(SUM(net_revenue), 0) AS total_revenue, COUNT(order_id) AS total_orders FROM {table_ref}"}]
}

dashboard_name = f"Challenge 6 Dashboard ({clean_username})"
resp = requests.post(
    f"https://{workspace_url}/api/2.0/lakeview/dashboards",
    headers=headers,
    json={"display_name": dashboard_name, "serialized_dashboard": json.dumps(dashboard_def), "parent_path": parent_path}
)

if resp.status_code == 200:
    dashboard_id = resp.json()["dashboard_id"]
    # Publish it
    requests.post(f"https://{workspace_url}/api/2.0/lakeview/dashboards/{dashboard_id}/published", headers=headers, json={"embed_credentials": True})
    print(f"\u2705 Dashboard created and published: {dashboard_name}")
    print(f"   Dashboard ID: {dashboard_id}")
else:
    print(f"\u26a0\ufe0f Dashboard create returned {resp.status_code}: {resp.text[:200]}")
    dashboard_id = "<CHECK_WORKSPACE>"

In [0]:
# --- Summary ---
print("="*60)
print("CHALLENGE 6 SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Tables:")
print(f"  gold_sales     - 1500 rows")
print(f"  gold_customers - 100 rows")
print(f"")
print(f"Dashboard:")
print(f"  Name: {dashboard_name}")
print(f"  ID:   {dashboard_id}")
print(f"  Status: PUBLISHED")
print(f"")
print(f"Full table paths:")
print(f"  {catalog}.{schema}.gold_sales")
print(f"  {catalog}.{schema}.gold_customers")
print(f"")
print(f"\u2b50 Save this dashboard_id for Task 5: {dashboard_id}")